# Hybrid Moderation Engine (Notebook)

A compact, single-notebook view of the hybrid content moderation MVP.
- Python 3.11 friendly and runnable end-to-end.
- Regex-only configuration with cached compilation for speed.
- Optional ML label scores that merge with rule outputs.
- Lightweight spam heuristics (URLs, suspicious TLDs, repetition, uppercase).
- English-only defaults and no external integrations.
- Decisions include action, label, score, rationale, rule_hits, and category_hits.


## Config schema and loading example


In [ ]:
from __future__ import annotations
import json
import copy
import re
from pathlib import Path
from functools import lru_cache
from typing import Any, Dict, List, Optional

LABELS = ['safe', 'harmful', 'false_information', 'spam', 'edge_cases']

DEFAULT_CONFIG: Dict[str, Any] = {
    'labels': LABELS,
    'preprocessing': {'lowercase': True, 'strip_urls': True, 'normalize_whitespace': True},
    'allowlist': ['thank you', 'have a nice day'],
    'denylist': ['kill', 'hate'],
    'category_patterns': {
        'harmful': [r'\bhate\b', r'\bkill(ing)?\b', r'\battack\b'],
        'false_information': [r'flat earth', r'moon landing was fake'],
        'spam': [r'free money', r'visit .* now'],
    },
    'spam': {
        'max_link_count': 2,
        'repetition_threshold': 3,
        'uppercase_ratio': 0.6,
        'url_presence_score': 0.4,
        'suspicious_tlds': ['ru', 'cn', 'tk', 'info'],
    },
    'arbitration': {'tie_margin': 0.05, 'min_label_score': 0.35, 'default_label': 'safe'},
}


def _deep_update(base: Dict[str, Any], updates: Dict[str, Any]) -> Dict[str, Any]:
    merged = copy.deepcopy(base)
    for key, value in updates.items():
        if isinstance(value, dict) and isinstance(merged.get(key), dict):
            merged[key] = _deep_update(merged[key], value)
        else:
            merged[key] = copy.deepcopy(value)
    return merged


def load_config(path: Optional[str] = None, overrides: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    config = copy.deepcopy(DEFAULT_CONFIG)
    if path:
        config_path = Path(path)
        if config_path.exists():
            with config_path.open('r', encoding='utf-8') as f:
                file_config = json.load(f)
            config = _deep_update(config, file_config)
    if overrides:
        config = _deep_update(config, overrides)
    return config


example_config = load_config()
example_config['labels'], example_config['arbitration']


## Preprocessing


In [ ]:
def preprocess_text(text: str, options: Dict[str, bool]) -> str:
    processed = text
    if options.get('lowercase', False):
        processed = processed.lower()
    if options.get('strip_urls', False):
        processed = re.sub(r'https?://\S+|www\.\S+', '', processed)
    if options.get('normalize_whitespace', False):
        processed = re.sub(r'\s+', ' ', processed).strip()
    return processed


## Rules engine


In [ ]:
class RuleResult:
    def __init__(self, label_scores: Dict[str, float], reasons: List[str], hard_label: Optional[str] = None,
                 rule_hits: Optional[List[str]] = None, category_hits: Optional[List[Dict[str, Any]]]=None):
        self.label_scores = label_scores
        self.reasons = reasons
        self.hard_label = hard_label
        self.rule_hits = rule_hits or []
        self.category_hits = category_hits or []

    def to_dict(self) -> Dict[str, Any]:
        data = {
            'label_scores': self.label_scores,
            'reasons': self.reasons,
            'rule_hits': self.rule_hits,
            'category_hits': self.category_hits,
        }
        if self.hard_label:
            data['hard_label'] = self.hard_label
        return data


@lru_cache(maxsize=128)
def _compile_pattern(pattern: str) -> re.Pattern[str]:
    return re.compile(pattern, flags=re.IGNORECASE)


def _uppercase_ratio(text: str) -> float:
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    uppercase = [c for c in letters if c.isupper()]
    return len(uppercase) / len(letters)


def evaluate_rules(text: str, config: Dict[str, Any], raw_text: Optional[str] = None) -> RuleResult:
    label_scores = {label: 0.0 for label in config.get('labels', LABELS)}
    reasons: List[str] = []
    rule_hits: List[str] = []
    category_hits: List[Dict[str, Any]] = []

    for pattern in config.get('allowlist', []):
        compiled = _compile_pattern(pattern)
        if compiled.search(text):
            label_scores['safe'] = 1.0
            reasons.append(f'allowlist pattern: {pattern}')
            rule_hits.append(f'allowlist::{pattern}')
            return RuleResult(label_scores, reasons, hard_label='safe', rule_hits=rule_hits, category_hits=category_hits)

    for pattern in config.get('denylist', []):
        compiled = _compile_pattern(pattern)
        if compiled.search(text):
            label_scores['harmful'] = 1.0
            reasons.append(f'denylist pattern: {pattern}')
            rule_hits.append(f'denylist::{pattern}')
            return RuleResult(label_scores, reasons, hard_label='harmful', rule_hits=rule_hits, category_hits=category_hits)

    for label, patterns in config.get('category_patterns', {}).items():
        for pattern in patterns:
            compiled = _compile_pattern(pattern)
            if compiled.search(text):
                label_scores[label] = max(label_scores[label], 0.8)
                reasons.append(f'pattern matched for {label}: {pattern}')
                category_hits.append({'label': label, 'pattern': pattern})

    spam_conf = config.get('spam', {})
    spam_source_text = raw_text if raw_text is not None else text
    urls = re.findall(r'https?://\S+|www\.\S+', spam_source_text)
    if urls:
        label_scores['spam'] = max(label_scores['spam'], float(spam_conf.get('url_presence_score', 0.4)))
        reasons.append('contains URLs')
        rule_hits.append('spam::url_presence')

    suspicious_tlds = spam_conf.get('suspicious_tlds', [])
    if suspicious_tlds and urls:
        lower_urls = [u.lower() for u in urls]
        if any(any(url.endswith(f'.{tld}') or f'.{tld}/' in url for tld in suspicious_tlds) for url in lower_urls):
            label_scores['spam'] = max(label_scores['spam'], 0.7)
            reasons.append('suspicious TLD detected')
            rule_hits.append('spam::suspicious_tld')

    if len(urls) > spam_conf.get('max_link_count', 2):
        label_scores['spam'] = max(label_scores['spam'], 0.9)
        reasons.append('excessive links detected')
        rule_hits.append('spam::link_count')

    tokens = text.split()
    if tokens:
        counts = {tok: tokens.count(tok) for tok in set(tokens)}
        if max(counts.values()) >= spam_conf.get('repetition_threshold', 3):
            label_scores['spam'] = max(label_scores['spam'], 0.85)
            reasons.append('repeated tokens detected')
            rule_hits.append('spam::repetition')

    uppercase_source = raw_text if raw_text is not None else text
    if _uppercase_ratio(uppercase_source) > spam_conf.get('uppercase_ratio', 0.6):
        label_scores['spam'] = max(label_scores['spam'], 0.7)
        reasons.append('high uppercase ratio')
        rule_hits.append('spam::uppercase_ratio')

    return RuleResult(label_scores, reasons, rule_hits=rule_hits, category_hits=category_hits)


## Hybrid arbiter


In [ ]:
def _merge_scores(rule_scores: Dict[str, float], ml_scores: Optional[Dict[str, float]]) -> Dict[str, float]:
    merged = dict(rule_scores)
    if ml_scores:
        for label, score in ml_scores.items():
            if label in merged:
                merged[label] = max(merged[label], float(score))
    return merged


def _top_two(scores: Dict[str, float]):
    sorted_pairs = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    top = sorted_pairs[0]
    second = sorted_pairs[1] if len(sorted_pairs) > 1 else ('', 0.0)
    return top, second


def _action_for_label(label: str) -> str:
    if label in ('harmful', 'spam'):
        return 'block'
    if label == 'safe':
        return 'allow'
    return 'review'


def arbitrate(rule_result: RuleResult, ml_scores: Optional[Dict[str, float]], config: Dict[str, Any]):
    if rule_result.hard_label:
        label = rule_result.hard_label
        return {
            'action': _action_for_label(label),
            'label': label,
            'score': 1.0,
            'confidence': 1.0,
            'label_scores': rule_result.label_scores,
            'rationale': rule_result.reasons,
            'rule_hits': rule_result.rule_hits,
            'category_hits': rule_result.category_hits,
            'source': 'rules',
        }

    merged_scores = _merge_scores(rule_result.label_scores, ml_scores)
    for label in config.get('labels', LABELS):
        merged_scores.setdefault(label, 0.0)

    top, second = _top_two(merged_scores)
    tie_margin = config.get('arbitration', {}).get('tie_margin', 0.05)
    min_label_score = config.get('arbitration', {}).get('min_label_score', 0.35)
    default_label = config.get('arbitration', {}).get('default_label', 'safe')

    if top[1] - second[1] < tie_margin:
        label = 'edge_cases'
        return {
            'action': _action_for_label(label),
            'label': label,
            'score': top[1],
            'confidence': top[1],
            'label_scores': merged_scores,
            'rationale': rule_result.reasons + ['scores within tie margin'],
            'rule_hits': rule_result.rule_hits,
            'category_hits': rule_result.category_hits,
            'source': 'hybrid',
        }

    if top[1] < min_label_score:
        label = default_label
        return {
            'action': _action_for_label(label),
            'label': label,
            'score': min_label_score,
            'confidence': min_label_score,
            'label_scores': merged_scores,
            'rationale': rule_result.reasons + ['below minimum score'],
            'rule_hits': rule_result.rule_hits,
            'category_hits': rule_result.category_hits,
            'source': 'hybrid',
        }

    label = top[0]
    return {
        'action': _action_for_label(label),
        'label': label,
        'score': top[1],
        'confidence': top[1],
        'label_scores': merged_scores,
        'rationale': rule_result.reasons + ['combined arbitration'],
        'rule_hits': rule_result.rule_hits,
        'category_hits': rule_result.category_hits,
        'source': 'hybrid',
    }


## Engine wrapper


In [ ]:
class HybridModerationEngine:
    def __init__(self, config_path: Optional[str] = None, overrides: Optional[Dict[str, Any]] = None):
        self.config = load_config(config_path, overrides)

    def _sanitize_scores(self, ml_scores: Optional[Dict[str, float]]):
        if ml_scores is None:
            return None
        allowed_labels = set(self.config.get('labels', LABELS))
        return {label: float(score) for label, score in ml_scores.items() if label in allowed_labels}

    def moderate(self, text: str, ml_scores: Optional[Dict[str, float]] = None) -> Dict[str, Any]:
        processed = preprocess_text(text, self.config.get('preprocessing', {}))
        rule_result = evaluate_rules(processed, self.config, raw_text=text)
        decision = arbitrate(rule_result, self._sanitize_scores(ml_scores), self.config)
        return {'input': text, 'processed': processed, 'decision': decision}


## Demo: three example inputs


In [ ]:
engine = HybridModerationEngine()
example_ml_scores = {'harmful': 0.52, 'safe': 0.5}

def run_demo(text: str, ml_scores: Optional[Dict[str, float]] = None):
    result = engine.moderate(text, ml_scores=ml_scores)
    print(json.dumps(result, indent=2))
    print('-' * 60)

print('Harmful denylist hit:')
run_demo('I will KILL you', ml_scores=None)

print('Spam leaning with suspicious TLD:')
run_demo('Visit www.example.cn now for free money!!!', ml_scores=None)

print('Tie margin review with close ML scores:')
run_demo('This is pretty neutral text', ml_scores=example_ml_scores)


## Optional: loading ML scores from a JSON file


In [ ]:
from tempfile import NamedTemporaryFile

def load_ml_scores_from_path(path: str) -> Dict[str, float]:
    with open(path, 'r', encoding='utf-8') as f:
        return {k: float(v) for k, v in json.load(f).items()}

sample_scores = {'spam': 0.6, 'safe': 0.3}
with NamedTemporaryFile('w+', suffix='.json', delete=True) as tmp:
    json.dump(sample_scores, tmp)
    tmp.flush()
    loaded_scores = load_ml_scores_from_path(tmp.name)

stdin_text = 'Check out www.promo.info for FREE MONEY'
print('CLI-style request with loaded scores:')
print(json.dumps(engine.moderate(stdin_text, ml_scores=loaded_scores), indent=2))
